In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [2]:
import os
import time
import random
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)


config_str = os.getenv("GPT_LUNA_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)

MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")


OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)

DATA_PATH = os.getenv(
    "DATA_PATH"
)

df=pd.read_csv(f'{DATA_PATH}df_n1.csv')

Using model: gpt-5.6-luna with reasoning effort: low


In [3]:

TOTAL_USAGE = {
    "prompt_tokens": 0,
    "reasoning_tokens": 0,
    "total_tokens": 0,
    "num_calls": 0,
}

def update_usage(usage):
    """
    Accumulate token usage across all LLM calls.
    """
    global TOTAL_USAGE

    u = usage.model_dump()

    TOTAL_USAGE["prompt_tokens"] += u.get("prompt_tokens", 0)
    TOTAL_USAGE["total_tokens"] += u.get("total_tokens", 0)
    TOTAL_USAGE["reasoning_tokens"] += (
        u.get("completion_tokens_details", {})
         .get("reasoning_tokens", 0)
    )
    TOTAL_USAGE["num_calls"] += 1
     
   

def build_context_messages(row):

    system_prompt = """
You are an expert software engineer performing context compression for an automated code review system.

Your task is NOT to summarize the file and NOT to reproduce the original file.
Your task is to extract the MINIMUM amount of source code required for another LLM to correctly review the given code hunk.

You will receive:
1. The original source file (<old_file>)
2. The changed code hunk (<hunk>)

Your goal:
Produce a compact code context that preserves only the information necessary to understand the behavior, dependencies, and potential issues of <hunk>.

Selection strategy:
- Start by including only the changed hunk.
- Add the smallest enclosing scope needed (function, method, or class).
- Add external definitions only when the hunk depends on them and their absence would make the behavior ambiguous.
- Add called functions, variables, attributes, imports, decorators, or parent classes only when they directly influence the logic of the hunk.
- Prefer short relevant snippets over complete files.
- If a definition is large, include only the relevant parts.

Strict exclusion rules:
- Do NOT return the entire file if it's loo large.
- Do NOT include file headers, licenses, comments, documentation, or unrelated code.
- Do NOT include neighboring functions unless they are required to understand the hunk.
- Do NOT include imports unless they are necessary to understand a referenced component.
- Do NOT include boilerplate code.

Think like a human reviewer:
A reviewer does not read the whole repository file. They inspect the changed code and only open the definitions required to reason about correctness.

Output requirements:
- Return ONLY the extracted source code.
- No explanations.
- No markdown fences.
- No comments about your selection process.

The final output should be significantly smaller than <old_file>.
"""

    user_prompt = f"""
<old_file>
{row["oldf"]}
</old_file>


<hunk>
{row["hunk"]}
</hunk>
"""

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


# ==========================
# Output parsing
# ==========================

def parse_context_output(text):

    if text is None:
        return ""

    return text.strip()


# ==========================
# OpenAI prediction
# ==========================

def predict_context(row):

    messages = build_context_messages(row)

    kwargs = {
        "model": MODEL_NAME,
        "messages": messages,
        "timeout": 120,
    }

    if REASONING_EFFORT:
        kwargs["reasoning_effort"] = REASONING_EFFORT

    response = client.chat.completions.create(**kwargs)
    
    update_usage(response.usage)

    
    raw_text = (
        response
        .choices[0]
        .message
        .content
    )

    context = parse_context_output(
        raw_text
    )

    return {
        "relevant_context": context
    }


# ==========================
# Pipeline execution
# ==========================

def run_context_pipeline(df):

    preds = []

    total = len(df)

    processed = 0

    base_sleep = SLEEP_BETWEEN_CALLS

    print(
        f"[START] Processing {total} rows"
    )

    for _, row in df.iterrows():

        processed += 1

        print(
            f"\n[ROW {processed}/{total}] Starting"
        )

        if (
            pd.isna(row["hunk"])
            or pd.isna(row["oldf"])
        ):

            print(
                "[SKIP] Missing old file or hunk"
            )

            preds.append(
                {
                    "relevant_context": ""
                }
            )

            continue

        success = False

        sleep_time = base_sleep

        for attempt in range(3):

            try:

                pred = predict_context(row)

                preds.append(
                    pred
                )

                success = True

                break

            except Exception as e:

                print(
                    f"[ERROR] {e}"
                )

                wait = (
                    sleep_time
                    + random.uniform(0, 1)
                )

                print(
                    f"[RETRY] waiting {wait:.2f}s"
                )

                time.sleep(wait)

                sleep_time *= 2

        if not success:

            preds.append(
                {
                    "relevant_context": ""
                }
            )

        wait = (
            base_sleep
            + random.uniform(0, 0.8)
        )

        time.sleep(wait)

        if processed % 10 == 0:

            print(
                f"[CHECKPOINT] processed {processed}/{total}"
            )

    print(
        "\n[DONE] Building dataframe"
    )

    pred_df = pd.DataFrame(
        preds
    )

    df = df.reset_index(
        drop=True
    )

    return pd.concat(
        [
            df,
            pred_df
        ],
        axis=1
    )

In [ ]:
df_result=run_context_pipeline(df)
df_relevant_context=df_result[['patch_id','relevant_context']]
df_relevant_context.to_csv(f'{DATA_PATH}df_n2_1.csv',index=False)

[START] Processing 1 rows

[ROW 1/1] Starting

[DONE] Building dataframe


In [6]:
import json
from pathlib import Path
from datetime import datetime
import tiktoken

# ==========================
# Token counting
# ==========================

encoding = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    if text is None:
        return 0
    return len(encoding.encode(str(text)))

# ==========================
# Compute compression statistics
# ==========================

total_old_tokens = df_result["oldf"].apply(count_tokens).sum()
total_generated_tokens = df_result["relevant_context"].apply(count_tokens).sum()

total_reduced_tokens = total_old_tokens - total_generated_tokens

compression_ratio = (
    total_generated_tokens / total_old_tokens
    if total_old_tokens else 0
)

reduction_ratio = 1 - compression_ratio

compression_factor = (
    total_old_tokens / total_generated_tokens
    if total_generated_tokens else 0
)

# ==========================
# Build log entry
# ==========================

log_entry = {
    "timestamp": datetime.now().isoformat(),
    "task": "context_compression (2)",
    "model": MODEL_NAME,
    "dataset_length": len(df_result),

    "context_statistics": {
        "total_original_context_tokens": int(total_old_tokens),
        "total_extracted_context_tokens": int(total_generated_tokens),
        "total_removed_tokens": int(total_reduced_tokens),
        "context_reduction_percentage": round(reduction_ratio * 100, 2),
        "compression_factor": round(compression_factor, 2),
    },

    "llm_usage": {
        "num_calls": TOTAL_USAGE["num_calls"],
        "prompt_tokens": TOTAL_USAGE["prompt_tokens"],
        "reasoning_tokens": TOTAL_USAGE["reasoning_tokens"],
        "total_tokens": TOTAL_USAGE["total_tokens"],
    }
}

# ==========================
# Append to log file
# ==========================

log_dir = Path("../logs")
log_dir.mkdir(exist_ok=True)

log_file = log_dir / "token_usage_insights_logs.json"

try:
    with open(log_file, "r", encoding="utf-8") as f:
        logs = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    logs = []

logs.append(log_entry)

with open(log_file, "w", encoding="utf-8") as f:
    json.dump(logs, f, indent=4)

print(f"Saved statistics to {log_file}")

Saved statistics to ..\logs\token_usage_insights_logs.json
